In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split as tts

df = pd.read_csv('abalone.data', header = None)

In [2]:
df[0] = df[0].map({'M':0,'F':1,'I':2})
x = df.iloc[:,:-1].values
y = df.iloc[:,-1].values.reshape(-1,1)

df.head()


,0,1,2,3,4,5,6,7,8
0,0,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15
1,0,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7
2,1,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9
3,0,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10
4,2,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7


In [3]:
x = (x - x.mean(axis=0)) / x.std(axis=0)

In [4]:
np.random.seed(42)
indices = np.random.permutation(len(x))
split = int(0.7 * len(x))

train_idx = indices[:split]
test_idx = indices[split:]

X_train, X_test = x[train_idx], x[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

In [5]:
class MLP:

    def __init__(self, input_size, hidden_size, output_size, lr=0.01, epochs=1000):
        self.lr = lr
        self.epochs = epochs

        self.W1 = np.random.randn(input_size, hidden_size)
        self.b1 = np.zeros((1, hidden_size))

        self.W2 = np.random.randn(hidden_size, output_size)
        self.b2 = np.zeros((1, output_size))

    def sigmoid(self, x):
        return 1/(1 + np.exp(-x))

    def sigmoid_derivative(self, x):
        return x * (1 - x)

    def forward(self, X):
        self.z1 = np.dot(X, self.W1) + self.b1
        self.a1 = self.sigmoid(self.z1)

        self.z2 = np.dot(self.a1, self.W2) + self.b2
        self.a2 = self.z2 

        return self.a2

    def backward(self, X, y):

        m = X.shape[0]
        dz2 = (self.a2 - y)

        dW2 = np.dot(self.a1.T, dz2) / m
        db2 = np.sum(dz2, axis=0, keepdims=True) / m

        dz1 = np.dot(dz2, self.W2.T) * self.sigmoid_derivative(self.a1)

        dW1 = np.dot(X.T, dz1) / m
        db1 = np.sum(dz1, axis=0, keepdims=True) / m

        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1

    def fit(self, X, y):

        for epoch in range(self.epochs):
            self.forward(X)
            self.backward(X, y)

            if epoch % 100 == 0:
                loss = np.mean((self.a2 - y) ** 2)
                print(f"Epoch {epoch}, Loss: {loss:.4f}")

    def predict(self, X):
        return self.forward(X)

In [6]:
model = MLP(input_size=X_train.shape[1], hidden_size=10, output_size=1)
model.fit(X_train, y_train)

Epoch 0, Loss: 133.5453
Epoch 100, Loss: 7.1235
Epoch 200, Loss: 6.6939
Epoch 300, Loss: 6.3826
Epoch 400, Loss: 6.1413
Epoch 500, Loss: 5.9537
Epoch 600, Loss: 5.8045
Epoch 700, Loss: 5.6826
Epoch 800, Loss: 5.5809
Epoch 900, Loss: 5.4948


In [7]:
y_pred = model.predict(X_test)

mse = np.mean((y_pred - y_test) ** 2)
print("Test MSE:", mse)

Test MSE: 4.749383917438826
